In [1]:
import os
os.chdir(r"C:\Users\Admin\Desktop\hospoda.CZECH-PUB\code\czech-pub-copy-to-load\czech-pub")

In [2]:
from __future__ import annotations

from utils.helpers import load_dataset, normalize

import json
from pathlib import Path
from collections import Counter, defaultdict

In [3]:
# data and file paths
DATA_PATH = Path.cwd() / "data/information-structure/generated"

FILES = [
    "inf-structure-baseline-80-raw.json",
    "inf-structure-explicit-question-80-raw.json",
    "inf-structure-correction-80-raw.json",
]  


## data analysis & check for missing values

In [4]:
all_data = []

for file in FILES:
    p = DATA_PATH / file
    data = load_dataset(p)

    print(f"loaded {file}: {len(data)} scenarios")
    all_data.extend(data)

print("\ntotal scenarios:", len(all_data))

loaded inf-structure-baseline-80-raw.json: 80 scenarios
loaded inf-structure-explicit-question-80-raw.json: 80 scenarios
loaded inf-structure-correction-80-raw.json: 80 scenarios

total scenarios: 240


In [5]:
# cue distribution
cue_counts = Counter(row.get("cue") for row in all_data)
print("Cue counts:", dict(cue_counts))

# do those fields exist: just to check
required_top = {"scenario_id", "cue", "seed"}
missing = []
for i, row in enumerate(all_data, start=1):
    missing_fields = required_top - set(row.keys())
    if missing_fields:
        missing.append((i, row.get("scenario_id"), sorted(missing_fields)))

print("Missing required top-level fields:", missing if missing else "NONE")

Cue counts: {'baseline': 80, 'explicit_question': 80, 'correction': 80}
Missing required top-level fields: NONE


In [6]:
# detect repeating svo triples in seed.base_proposition 


def find_repeated_triples(rows):

    triple_to_items = {}

    for idx, row in enumerate(rows, start=1):
        base = row.get("seed", {}).get("base_proposition", {})
        s, v, o = base.get("subject"), base.get("verb"), base.get("object")
        if s is None or v is None or o is None:
            raise ValueError(f"none in the triples.")

        triple = (normalize(s), normalize(v), normalize(o))
        key = row.get("scenario_id")
        
        if triple not in triple_to_items:
            triple_to_items[triple] = []
        triple_to_items[triple].append(key)

    repeats = {t: ids for t, ids in triple_to_items.items() if len(ids) > 1}
    return repeats, triple_to_items

In [7]:
repeats, all_triples = find_repeated_triples(all_data)
print("unique:", len(all_triples), "repeated:", len(repeats))
for triple, ids in repeats.items():
    print(triple, "->", ids)

unique: 230 repeated: 10
('grafik', 'upravil', 'logo') -> [30, 35]
('skladník', 'vybalil', 'zásilku') -> [31, 2]
('barman', 'namíchal', 'drink') -> [32, 65]
('zahradník', 'ostříhal', 'keře') -> [38, 17]
('údržbář', 'namazal', 'zámek') -> [46, 6]
('automechanik', 'vypustil', 'nádrž') -> [48, 15]
('instalatér', 'utáhl', 'kohoutek') -> [50, 18]
('klára', 'ztratila', 'prsten') -> [62, 111]
('technik', 'restartoval', 'server') -> [64, 36]
('sekretářka', 'naskenovala', 'smlouvu') -> [12, 102]


## transform to master

In [8]:
def capitalize_first(text: str):
    text = " ".join(str(text).split())
    if not text:
        return text
    return text[0].upper() + text[1:]


def make_sentence(parts, ending="."):
    sentence = " ".join(str(p).strip() for p in parts if p is not None)
    sentence = capitalize_first(sentence)
    if sentence and sentence[-1] not in ".!?":
        sentence += ending
    return sentence


def unpack_basic_proposition(base: dict):
    s = base["subject"]
    v = base["verb"]
    o = base["object"]

    return {
        "SVO": make_sentence([s, v, o]),
        "VSO": make_sentence([v, s, o]),
        "OVS": make_sentence([o, v, s]),
        "SOV": make_sentence([s, o, v]),
    }


def object_last_structure_for_case(case: str):
    """
    object_last option:
    - focus_object -> SVO
    - focus_subject / focus_verb -> VSO
    """
    if case == "focus_object":
        return "SVO"
    if case in {"focus_subject", "focus_verb"}:
        return "VSO"
    raise ValueError(f"Unknown case: {case}")


def right_answer_for_case(case: str) -> str:
    """ map each case to the corresponding right answer """
    mapping = {
        "focus_object": "object_last",
        "focus_subject": "subject_last",
        "focus_verb": "verb_last",
    }
    return mapping[case]


def build_options(seed: dict, case: str) -> dict:
    """ builds answer options for the given case, extracting data from the scenairo's seed """
    variants = unpack_basic_proposition(seed["base_proposition"])
    object_last_structure = object_last_structure_for_case(case)

    return {
        "object_last": {
            "structure": object_last_structure,
            "sentence": variants[object_last_structure],
        },
        "subject_last": {
            "structure": "OVS",
            "sentence": variants["OVS"],
        },
        "verb_last": {
            "structure": "SOV",
            "sentence": variants["SOV"],
        },
        "distractor": {
            "sentence": seed["distractor_bank"][case],
        },
    }


def build_atomic_item(scenario: dict, case: str) -> dict:
    """ Builds an item for the given case from the scenario. includes building options"""
    seed = scenario["seed"]

    return {
        "id": f"{scenario['scenario_id']}-{case}",
        "scenario_id": scenario["scenario_id"],
        "case": case,
        "cue": scenario["cue"],
        "context": seed["contexts"][case],
        "utterance": seed["utterance"],
        "options": build_options(seed, case),
        "right_answer": right_answer_for_case(case),
    }


def expand_scenario_to_atomic_items(scenario: dict) -> list[dict]:
    cases = ["focus_object", "focus_subject", "focus_verb"]
    return [build_atomic_item(scenario, case) for case in cases]

In [9]:
def build_items_from_scenarios(scenarios: list[dict]) -> list[dict]:
    items = []

    for scenario in scenarios:
        atomic_items = expand_scenario_to_atomic_items(scenario)
        items.extend(atomic_items)

    return items

In [10]:
len(all_data)

240

In [11]:
all_items = build_items_from_scenarios(all_data)
len(all_items)

720

In [13]:
all_items[0]

{'id': '17-focus_object',
 'scenario_id': 17,
 'case': 'focus_object',
 'cue': 'baseline',
 'context': 'Z kuchyně bylo cítit, že kuchař sice něco připálil, ale nebylo jasné, o jaký pokrm vlastně šlo.',
 'utterance': 'Marek na to najednou prohlásil:',
 'options': {'object_last': {'structure': 'SVO',
   'sentence': 'Kuchař připálil omáčku.'},
  'subject_last': {'structure': 'OVS', 'sentence': 'Omáčku připálil kuchař.'},
  'verb_last': {'structure': 'SOV', 'sentence': 'Kuchař omáčku připálil.'},
  'distractor': {'sentence': 'Tato omáčka vyžaduje pro správnou chuť velmi čerstvé suroviny.'}},
 'right_answer': 'object_last'}

In [14]:
path = r"data\information-structure\master\information-structure-all-items-720.json"
with open(path, "w", encoding="utf-8") as f:
        json.dump(all_items, f, ensure_ascii=False, indent=2)

## adding labels: A,B,C,D

In [15]:
def add_option_labels(item: dict) -> dict:
    option_order = ["object_last", "subject_last", "verb_last", "distractor"]
    labels = ["A", "B", "C", "D"]

    for key, label in zip(option_order, labels):
        item["options"][key]["label"] = label

    return item

In [16]:
for item in all_items:
    item = add_option_labels(item)

## observe an example item

In [17]:
def render_item_preview(item: dict) -> str:
    lines = []

    lines.append(f"CONTEXT: {item['context']}")
    lines.append(f"UTTERANCE: {item['utterance']}")
    lines.append("OPTIONS:")

    option_order = ["object_last", "subject_last", "verb_last", "distractor"]

    for key in option_order:
        opt = item["options"][key]

        lines.append(f"- [{opt['label']}]: {opt['sentence']}")

    lines.append(f"\nRIGHT ANSWER: {item["options"][item["right_answer"]]["label"]}")

    return "\n".join(lines) 

In [18]:
print(render_item_preview(all_items[0]))

CONTEXT: Z kuchyně bylo cítit, že kuchař sice něco připálil, ale nebylo jasné, o jaký pokrm vlastně šlo.
UTTERANCE: Marek na to najednou prohlásil:
OPTIONS:
- [A]: Kuchař připálil omáčku.
- [B]: Omáčku připálil kuchař.
- [C]: Kuchař omáčku připálil.
- [D]: Tato omáčka vyžaduje pro správnou chuť velmi čerstvé suroviny.

RIGHT ANSWER: A


In [19]:
# convert to eval format

import json


def convert_information_structure_item(item: dict) -> dict:
    # for now its fixed
    option_order = ["object_last", "subject_last", "verb_last", "distractor"]
    default_labels = ["A", "B", "C", "D"]

    # Map internal option keys to shared-schema option types
    type_map = {
        "object_last": "object_last",
        "subject_last": "subject_last",
        "verb_last": "verb_last",
        "distractor": "distractor",
    }

    options_out = []
    label_by_option_key = {}

    for key, default_label in zip(option_order, default_labels):
        opt = item["options"][key]

        label = opt.get("label", default_label)

        options_out.append({
            "label": label,
            "type": type_map[key],
            "text": opt["sentence"],
        })

        label_by_option_key[key] = label

    correct_option_label = label_by_option_key[item["right_answer"]]

    converted = {
        "id": f'"information_structure"-{item['cue']}-{item["id"]}',
        "phenomenon": "information_structure",
        "category": f"{item['cue']}_{item['case']}",
        "context": item["context"],
        "utterance": item["utterance"],
        "question": "",
        "options": options_out,
        "correct_option": correct_option_label,
        "meta_general": {
            "creation_method": "generation",
            "corpus": "",
            "genre": "",
        },
        "meta_special": {
            "trigger": "",
            "negation": "",
        }
    }

    return converted